# ATE end-to-end examples (genriesz)

This notebook demonstrates how to estimate the **Average Treatment Effect (ATE)** with **genriesz**.

We assume the regressor has the form:

- `X = [D, Z...]`, where `D` is a **binary treatment indicator** (`0/1`),
- `Y` is the observed outcome.

We will compute (optionally with cross-fitting):

- **RA**: regression adjustment (plug-in)
- **RW**: Riesz weighting (weighting only)
- **ARW**: augmented Riesz weighting
- **TMLE**: targeted minimum loss estimation (one-step fluctuation)

We also show how to swap the **basis**:
- polynomial features,
- RKHS-style RBF random features,
- nearest-neighbor matching (kNN catchment basis),
- random forest leaf features (optional),
- neural network embeddings (optional).


In [1]:
import numpy as np

from genriesz import (
    grr_ate,
    SquaredGenerator,
    UKLGenerator,
    PolynomialBasis,
    TreatmentInteractionBasis,
    RBFRandomFourierBasis,
    KNNCatchmentBasis,
)

rng = np.random.default_rng(0)


## Synthetic data

In [2]:
# Data-generating process
n = 3000
d_z = 5

Z = rng.normal(size=(n, d_z))

# Treatment assignment: e(Z) = sigmoid(a'Z)
logits = 0.7 * Z[:, 0] - 0.3 * Z[:, 1]
e = 1.0 / (1.0 + np.exp(-logits))
D = rng.binomial(1, e, size=n).astype(float)

# Potential outcomes (constant effect for simplicity)
tau = 1.0
mu0 = 0.5 * Z[:, 0] + 0.25 * Z[:, 1] ** 2
Y0 = mu0 + rng.normal(scale=1.0, size=n)
Y1 = mu0 + tau + rng.normal(scale=1.0, size=n)

Y = D * Y1 + (1.0 - D) * Y0

# Regressor matrix X = [D, Z...]
X = np.column_stack([D, Z])

print("X shape:", X.shape, "Y shape:", Y.shape)


X shape: (3000, 6) Y shape: (3000,)


## Example 1: Polynomial basis + treatment interactions

In [3]:
# Basis on Z, then interact with D (ATE-friendly)
psi = PolynomialBasis(degree=2, include_bias=True)
phi = TreatmentInteractionBasis(base_basis=psi)

# Generator: Squared loss (always safe / no domain constraints)
gen = SquaredGenerator(C=0.0).as_generator()

res_poly = grr_ate(
    X=X,
    Y=Y,
    basis=phi,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res_poly.summary_text())


ATE estimates (n=3000)
alpha=0.05 | null=0.0
diagnostics: max_abs_smd_unweighted=0.7095986621457625, max_abs_smd_weighted=0.016620078977766132, ess_treated=1188.4096325793273, ess_control=1260.0346792242683

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                 1.03046    0.00359346         [ 1.02342,  1.03751]           0
RW                 1.03302     0.0569279         [ 0.921445,  1.1446]           0
ARW                1.02916     0.0419132        [ 0.947017,  1.11131]           0
TMLE               1.02921     0.0419128        [ 0.947063,  1.11136]           0


## Example 2: RKHS-style basis (RBF random Fourier features)

This approximates an RBF kernel feature map using random Fourier features, then
interacts the features with treatment.


In [4]:
psi_rff = RBFRandomFourierBasis(
    n_features=500,
    sigma=1.0,
    standardize=True,
    random_state=0,
)
phi_rff = TreatmentInteractionBasis(base_basis=psi_rff)

res_rff = grr_ate(
    X=X,
    Y=Y,
    basis=phi_rff,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res_rff.summary_text())


ATE estimates (n=3000)
alpha=0.05 | null=0.0
diagnostics: max_abs_smd_unweighted=0.7095986621457625, max_abs_smd_weighted=0.27862992844919376, ess_treated=1363.470230442599, ess_control=1370.9575124421833

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                 1.15729    0.00717955         [ 1.14321,  1.17136]           0
RW                 1.22541     0.0540494         [ 1.11947,  1.33134]           0
ARW                1.12627     0.0423592         [ 1.04325,  1.20929]           0
TMLE               1.12732     0.0423287         [ 1.04436,  1.21028]           0


## Example 3: Nearest-neighbor matching (kNN catchment-area basis)

Nearest-neighbor matching can be expressed using a **catchment-area indicator basis**

\[
\phi_j(z) = \mathbf{1}\{c_j \in \mathrm{NN}_k(z)\},
\]

and is shown in the paper to be a special case of squared-loss Riesz regression.

Below we compute a matching-style ATE estimate using the catchment basis directly.
For a fully general GRR workflow, you can also pass the catchment basis as `basis=...`
to `grr_ate`.


In [5]:
# Matching-style estimate with a kNN catchment basis
# (This mirrors examples/ate_synthetic_nn_matching.py.)

Z_only = X[:, 1:]  # drop D
n_centers = 400

centers = Z_only[:n_centers]
queries = Z_only[n_centers:]

basis_knn = KNNCatchmentBasis(n_neighbors=1).fit(centers)

# For each query point, w_j counts how often center j is selected
Phi = basis_knn(queries)  # (n_queries, n_centers), dense 0/1
w = Phi.sum(axis=0)       # (n_centers,)

# Matching estimator for ATE:
#   theta_hat = mean_{treated} Y - sum_{controls} w_i Y_i / n_treated
D_cent = D[:n_centers]
Y_cent = Y[:n_centers]

treated = (D_cent == 1)
control = (D_cent == 0)

n_treated = treated.sum()
ate_match = float(Y_cent[treated].mean() - (w[control] @ Y_cent[control]) / n_treated)

print("Matching-style ATE estimate:", ate_match)


Matching-style ATE estimate: 0.24697367793676572


## Example 4: Random forest leaf basis (optional)

If you have `scikit-learn` installed, you can use a random forest as a **feature map**
via leaf indicators. This keeps GRR convex (linear in parameters) while giving a
flexible nonparametric basis.


In [6]:
from sklearn.ensemble import RandomForestRegressor
from genriesz.sklearn_basis import RandomForestLeafBasis

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=6,
    random_state=0,
)

leaf_basis = RandomForestLeafBasis(rf, include_bias=True).fit(X, Y)

res_rf = grr_ate(
    X=X,
    Y=Y,
    basis=leaf_basis,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res_rf.summary_text())

ATE estimates (n=3000)
alpha=0.05 | null=0.0
diagnostics: max_abs_smd_unweighted=0.7095986621457625, max_abs_smd_weighted=0.786895849078183, ess_treated=350.8795988622065, ess_control=507.56068177539834

Estimator         Estimate            SE                           CI     p-value
---------------------------------------------------------------------------------
RA                 1.03187     0.0199079        [ 0.992854,  1.07089]           0
RW                 1.69371      0.252221         [ 1.19937,  2.18806]    1.88e-11
ARW                1.25382      0.215261        [ 0.831919,  1.67573]    5.72e-09
TMLE                 1.063      0.217126         [ 0.63744,  1.48856]    9.79e-07


## Example 5: Neural network embedding basis (optional)

If you have PyTorch installed, you can use a neural network as a **fixed feature map**.
A recommended workflow is:

1. train an embedding network on a separate task,
2. freeze it,
3. use its outputs as features in GRR.

Below we show the mechanics with a small MLP (training is optional).


In [ ]:
import torch
from genriesz.torch_basis import MLPEmbeddingNet, TorchEmbeddingBasis

torch.manual_seed(0)

net = MLPEmbeddingNet(input_dim=X.shape[1], hidden_dims=(64,), output_dim=32)
# (Optional) Train net here on a separate task.
# For a lightweight demo, we skip training and just use the random initialization.
nn_basis = TorchEmbeddingBasis(net, include_bias=True, device="cpu")

res_nn = grr_ate(
    X=X,
    Y=Y,
    basis=nn_basis,
    generator=gen,
    cross_fit=True,
    folds=5,
    random_state=0,
    estimators=("ra", "rw", "arw", "tmle"),
    outcome_models="shared",
    riesz_penalty="l2",
    riesz_lam=1e-3,
    max_iter=300,
    tol=1e-8,
)

print(res_nn.summary_text())

## Generator / regularization sweep (SQ / UKL / BP)

This section runs **SQ-Riesz**, **UKL-Riesz**, and **BP-Riesz** for the same
polynomial interaction basis, and compares multiple regularization norms and
strengths.

We use a **branch function** for UKL/BP that forces:

- positive branch for treated units (`D=1`),
- negative branch for control units (`D=0`).

All four estimators (**RA / RW / ARW / TMLE**) are reported.

In [ ]:
from genriesz import BPGenerator

branch = lambda x: int(x[0] == 1.0)

generator_grid = [
    ("SQ", SquaredGenerator(C=0.0).as_generator()),
    ("UKL (C=1)", UKLGenerator(C=1.0, branch_fn=branch).as_generator()),
    ("BP (omega=0.1, C=1)", BPGenerator(C=1.0, omega=0.1, branch_fn=branch).as_generator()),
    ("BP (omega=0.2, C=1)", BPGenerator(C=1.0, omega=0.2, branch_fn=branch).as_generator()),
    ("BP (omega=0.5, C=1)", BPGenerator(C=1.0, omega=0.5, branch_fn=branch).as_generator()),
]

penalty_grid = [
    {"penalty": "l2", "lam": 1e-4, "p_norm": None},
    {"penalty": "l2", "lam": 1e-3, "p_norm": None},
    {"penalty": "l1", "lam": 1e-4, "p_norm": None},
    {"penalty": "l1", "lam": 1e-3, "p_norm": None},
    {"penalty": "lp", "lam": 1e-4, "p_norm": 1.5},
    {"penalty": "lp", "lam": 1e-3, "p_norm": 1.5},
]

rows = []
for gname, gen_i in generator_grid:
    for cfg in penalty_grid:
        row = {
            "generator": gname,
            "penalty": cfg["penalty"],
            "lam": cfg["lam"],
            "p_norm": cfg["p_norm"],
            "status": "ok",
            "message": "",
        }

        try:
            res_i = grr_ate(
                X=X,
                Y=Y,
                basis=phi,
                generator=gen_i,
                cross_fit=True,
                folds=3,
                random_state=0,
                estimators=("ra", "rw", "arw", "tmle"),
                outcome_models="shared",
                outcome_link="identity",
                riesz_penalty=cfg["penalty"],
                riesz_lam=cfg["lam"],
                riesz_p_norm=cfg["p_norm"],
                max_iter=250,
                tol=1e-8,
            )

            for k in ("ra", "rw", "arw", "tmle"):
                e = res_i.estimates[k]
                row[k] = e.estimate
                row[f"{k}_se"] = e.se
                row[f"{k}_err"] = e.estimate - tau  # tau is the true constant effect in this DGP

        except Exception as err:
            row["status"] = "failed"
            row["message"] = str(err)
            for k in ("ra", "rw", "arw", "tmle"):
                row[k] = np.nan
                row[f"{k}_se"] = np.nan
                row[f"{k}_err"] = np.nan

        rows.append(row)

import pandas as pd

df = pd.DataFrame(rows)
# Sort by absolute ARW error (ARW is typically stable)
df = df.sort_values(by="arw_err", key=lambda s: np.abs(s))
display(df)
